Imports y rutas

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob

DATA_FEATURES_DIR = Path("../../data/features/hotel_ttoo")

files = glob.glob(str(DATA_FEATURES_DIR / "*.parquet"))
df_original = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
df_original["fecha"] = pd.to_datetime(df_original["fecha"], unit="ms")
df_original["hotel"] = df_original["hotel"].str.upper()

IMPORTANCE_DIR = Path("../../models/importances")
DASHBOARD_DATA_DIR = Path("../../dashboard/data")

DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

Carga del SHAP temporal XGBoost

In [11]:
shap_temporal_path = IMPORTANCE_DIR / "xgb_shap_temporal.csv"

df_shap_temp = pd.read_csv(shap_temporal_path, parse_dates=["fecha"])

print("Shape SHAP temporal:", df_shap_temp.shape)
display(df_shap_temp.head())

Shape SHAP temporal: (60760, 7)


,fecha,hotel,year,canal,ocup_total,shap_value,family
0,2023-01-01,HOTEL_1,2023,is_weekend,0.510638,-0.000753,calendario
1,2023-01-02,HOTEL_1,2023,is_weekend,0.659574,0.000197,calendario
2,2023-01-03,HOTEL_1,2023,is_weekend,0.627660,0.000609,calendario
3,2023-01-04,HOTEL_1,2023,is_weekend,0.670213,0.000632,calendario
4,2023-01-05,HOTEL_1,2023,is_weekend,0.691489,0.003358,calendario


Limpieza y validación básica

In [12]:
df_shap_temp["hotel"] = df_shap_temp["hotel"].str.upper()

df_shap_temp["canal"] = np.where(
    df_shap_temp["canal"].str.startswith("rn_") | 
    ~df_shap_temp["canal"].str.startswith(("season_", "is_")),
    df_shap_temp["canal"].str.upper(),   # TTOO → mayúsculas
    df_shap_temp["canal"]                # season_ / is_weekend → sin tocar
)

df_shap_temp = df_shap_temp.dropna(subset=["shap_value"])

df_shap_temp = df_shap_temp.sort_values(["hotel", "canal", "fecha"])

print("Shape limpio:", df_shap_temp.shape)
display(df_shap_temp.groupby("hotel").head(3))

Shape limpio: (60760, 7)


,fecha,hotel,year,canal,ocup_total,shap_value,family
1024,2023-01-01,HOTEL_1,2023,AL,0.510638,-0.000330,ttoo
1025,2023-01-02,HOTEL_1,2023,AL,0.659574,-0.000199,ttoo
1026,2023-01-03,HOTEL_1,2023,AL,0.627660,-0.000233,ttoo
21409,2023-01-01,HOTEL_2,2023,AL,0.811594,0.000000,ttoo
21410,2023-01-02,HOTEL_2,2023,AL,0.826087,0.000000,ttoo
21411,2023-01-03,HOTEL_2,2023,AL,0.782609,0.000000,ttoo
40145,2023-01-01,HOTEL_3,2023,AL,0.613546,-0.001965,ttoo
40146,2023-01-02,HOTEL_3,2023,AL,0.653386,-0.000664,ttoo
40147,2023-01-03,HOTEL_3,2023,AL,0.633466,-0.007402,ttoo


Atributos temporales (clave para BI)

In [13]:
df_shap_temp["year"] = df_shap_temp["fecha"].dt.year
df_shap_temp["month"] = df_shap_temp["fecha"].dt.month
df_shap_temp["week"] = df_shap_temp["fecha"].dt.isocalendar().week.astype(int)
df_shap_temp["dow"] = df_shap_temp["fecha"].dt.dayofweek
df_shap_temp["dayofyear"] = df_shap_temp["fecha"].dt.dayofyear

display(df_shap_temp.head())

,fecha,hotel,year,canal,ocup_total,shap_value,family,month,week,dow,dayofyear
1024,2023-01-01,HOTEL_1,2023,AL,0.510638,-0.000330,ttoo,1,52,6,1
1025,2023-01-02,HOTEL_1,2023,AL,0.659574,-0.000199,ttoo,1,1,0,2
1026,2023-01-03,HOTEL_1,2023,AL,0.627660,-0.000233,ttoo,1,1,1,3
1027,2023-01-04,HOTEL_1,2023,AL,0.670213,-0.000274,ttoo,1,1,2,4
1028,2023-01-05,HOTEL_1,2023,AL,0.691489,0.000204,ttoo,1,1,3,5


Reconstruir season_autumn como categoría base

In [14]:
# ── Reconstruir season_autumn como categoría base ──────────────────────────
# Su SHAP = -(suma de las otras 3 estaciones) para que el efecto neto sea 0
autumn_rows = []

for (hotel, fecha), grp in df_shap_temp[df_shap_temp["family"] == "season"].groupby(["hotel", "fecha"]):
    
    shap_otras = grp["shap_value"].sum()
    
    # Fila base: otoño = negativo de la suma de las otras 3
    row = grp.iloc[0].copy()
    row["canal"]      = "season_autumn"
    row["family"]     = "season"
    row["shap_value"] = -shap_otras
    
    autumn_rows.append(row)

df_autumn = pd.DataFrame(autumn_rows)
df_shap_temp = pd.concat([df_shap_temp, df_autumn], ignore_index=True)
df_shap_temp = df_shap_temp.sort_values(["hotel", "canal", "fecha"]).reset_index(drop=True)

print("Verificación seasons:")
display(df_shap_temp.groupby(["hotel","family"])["canal"].nunique().reset_index())

Verificación seasons:


,hotel,family,canal
0,HOTEL_1,calendario,1
1,HOTEL_1,season,4
2,HOTEL_1,ttoo,16
3,HOTEL_2,calendario,1
4,HOTEL_2,season,4
5,HOTEL_2,ttoo,16
6,HOTEL_3,calendario,1
7,HOTEL_3,season,4
8,HOTEL_3,ttoo,16


Normalización ligera para visualización

In [15]:
df_shap_temp["shap_norm"] = (
    df_shap_temp
    .groupby(["hotel", "canal"])["shap_value"]
    .transform(lambda x: x / (np.abs(x).quantile(0.95) + 1e-6))
)

display(df_shap_temp.head())

,fecha,hotel,year,canal,ocup_total,shap_value,family,month,week,dow,dayofyear,shap_norm
0,2023-01-01,HOTEL_1,2023,AL,0.510638,-0.000330,ttoo,1,52,6,1,-0.084229
1,2023-01-02,HOTEL_1,2023,AL,0.659574,-0.000199,ttoo,1,1,0,2,-0.050718
2,2023-01-03,HOTEL_1,2023,AL,0.627660,-0.000233,ttoo,1,1,1,3,-0.059284
3,2023-01-04,HOTEL_1,2023,AL,0.670213,-0.000274,ttoo,1,1,2,4,-0.069875
4,2023-01-05,HOTEL_1,2023,AL,0.691489,0.000204,ttoo,1,1,3,5,0.052098


Exportación final para dashboard

In [16]:
# ── Verificación rápida ────────────────────────────────────────────────────
print("\nFamilias presentes:")
display(df_shap_temp.groupby(["hotel", "family"])["canal"].nunique().reset_index())


Familias presentes:


,hotel,family,canal
0,HOTEL_1,calendario,1
1,HOTEL_1,season,4
2,HOTEL_1,ttoo,16
3,HOTEL_2,calendario,1
4,HOTEL_2,season,4
5,HOTEL_2,ttoo,16
6,HOTEL_3,calendario,1
7,HOTEL_3,season,4
8,HOTEL_3,ttoo,16


In [17]:
df_flags = df_original[["fecha", "hotel", "season", "is_weekend"]].drop_duplicates()
df_shap_temp = df_shap_temp.merge(df_flags, on=["fecha", "hotel"], how="left")

In [18]:
out_path = DASHBOARD_DATA_DIR / "xgb_shap_temporal.parquet"

df_shap_temp.to_parquet(out_path, index=False)

print("✅ Dataset SHAP temporal exportado a:", out_path)

✅ Dataset SHAP temporal exportado a: ..\..\dashboard\data\xgb_shap_temporal.parquet
